# Домашнее задание №3: DNABERT-2, LoRA и Reinforcement Learning для генерации промоторов

В этом задании вы построите двухэтапный пайплайн для работы с ДНК-последовательностями:

1. **Supervised learning:** файнтюнинг `DNABERT-2` как бинарного классификатора промотор / не-промотор, а также сравнение полного fine-tuning с `LoRA`.
2. **Reinforcement learning:** обучение агента, который пошагово генерирует ДНК-последовательность длины 300, получая награду от обученного классификатора.

Идея задания — не просто запустить готовые библиотеки, а понять, как связаны:
- представления последовательностей в трансформерах;
- parameter-efficient fine-tuning;
- sparse-reward RL в задаче генерации.

![Пайплайн ДЗ 3](images/pipeline.png)

## Оглавление
- [Правила игры и оценивание](#rules)
- [0. Установка и настройка](#setup)
- [1. Биологические вопросы](#bio-questions)
- [2. Трансформеры в геномике (теория)](#transformers-theory)
  - [2.1 Вопросы по DNABERT-2](#dnabert2-questions)
- [3. Fine-tuning DNABERT-2](#finetuning)
  - [3.1 Загрузка и исследование данных](#data-loading)
  - [3.2 Токенизация и DataLoader](#tokenization)
  - [3.3 Модель с classification head](#model)
  - [3.4 Функции обучения](#training-functions)
  - [3.5 Обучающий цикл](#training-loop)
  - [3.6 Оценка модели](#evaluation)
  - [3.7 LoRA Fine-tuning](#lora)
- [4. Введение в Reinforcement Learning](#rl-theory)
- [5. RL-среда](#rl-env)
  - [5.1 Класс DNAEnvironment](#dna-env)
  - [5.2 Функция награды](#reward-fn)
- [6. Policy Network — LSTMPolicy](#lstm-policy)
- [7. Алгоритм REINFORCE](#reinforce)
  - [7.1 Сбор траектории](#trajectory)
  - [7.2 Baseline и Advantage](#baseline)
  - [7.3 Обновление политики](#policy-update)
- [8. Базовые стратегии](#baselines)
  - [8.1 Случайная генерация](#random-baseline)
  - [8.2 Частотный baseline](#freq-baseline)
- [9. Обучение и анализ](#training-analysis)
  - [9.1 Обучающий цикл REINFORCE](#rl-training)
  - [9.2 Визуализация обучения](#visualization)
  - [9.3 Анализ последовательностей](#seq-analysis)
  - [9.4 Дискуссионные вопросы](#discussion)
- [Бонус: PPO](#ppo)

---

<a id="rules"></a>
## Правила игры и оценивание

Чтобы задание было полезным, а не превратилось в копипасту, придерживайтесь следующих принципов.

#### 1. Использование AI-ассистентов

Вы можете использовать LLM (ChatGPT, Claude и др.), но по прозрачным правилам:

- **Теоретические вопросы.**  
  Сначала вставьте «сырой» ответ ИИ, а ниже обязательно напишите **свой** ответ, перефразировав и дополнив его. Без собственного текста теоретический блок не засчитывается.
- **Практический код.**  
  Если вы использовали ИИ для написания фрагмента кода, объясните:
  - что делает этот код;
  - почему он устроен именно так;
  - какие у него ограничения и какие возможны альтернативы.

#### 2. Самостоятельная реализация

- **Разрешено:** менять гиперпараметры, улучшать структуру кода, добавлять собственные проверки, дополнительные графики и анализ.
- **Нежелательно:** превращать решение в набор чёрных ящиков без понимания. Цель задания — руками собрать пайплайн fine-tuning + RL и понимать, как он работает.

#### 3. Что считается хорошим результатом

Минимально ожидается, что:
- классификатор на основе DNABERT-2 обучается и показывает разумное качество на тесте;
- RL-агент обучается без ошибок и превосходит случайную генерацию;
- в отчёте есть осмысленный анализ ограничений reward-модели и baseline-стратегий.

#### 4. Формат сдачи

Отправьте `.ipynb`-ноутбук:
- со всеми выполненными ячейками;
- с сохранёнными output;
- с заполненными текстовыми ответами;
- с графиками и краткими выводами по экспериментам.

### Оценивание

| Раздел | Баллы |
| :-- | :--: |
| Биологические вопросы | 2.5 |
| Вопросы по DNABERT-2 | 2.5 |
| Fine-tuning DNABERT-2 | 15 |
| LoRA Fine-tuning | 10 |
| RL-среда | 15 |
| LSTMPolicy | 10 |
| REINFORCE | 20 |
| Базовые стратегии | 10 |
| Обучение и анализ | 15 |
| **Итого** | **100** |
| **Бонус: PPO** | **+10** |

> **Важно:** если не ответить на теоретические вопросы или не привести свой текст после ответа ИИ, теоретическая часть не засчитывается.

<a id="setup"></a>

## Раздел 0. Установка и настройка

In [ ]:
!pip install -q transformers datasets evaluate scikit-learn matplotlib seaborn numpy tqdm logomaker
print("Библиотеки установлены")

In [ ]:
import os, json, random, math          # стандартные модули: работа с ОС, JSON, рандомом и базовой математикой
import numpy as np                     # NumPy для работы с массивами и численной статистикой
import matplotlib.pyplot as plt        # Matplotlib для построения графиков
import seaborn as sns                  # Seaborn для более удобной визуализации и стилизации графиков
from tqdm.auto import tqdm             # tqdm для прогресс-баров в циклах (в том числе в ноутбуках)

import torch                           # базовый PyTorch (тензоры, автодифф, устройство)
import torch.nn as nn                  # модуль nn: слои нейросетей, контейнеры моделей
import torch.nn.functional as F        # функциональный API: активации, потери и др. без параметров
from torch.utils.data import DataLoader, Dataset  # абстракции Dataset/DataLoader для батчевой загрузки данных

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
# AutoTokenizer / AutoModel — загрузка токенизатора и модели DNABERT-2 из HuggingFace Hub
# get_linear_schedule_with_warmup — линейный scheduler для learning rate (warmup + decay)

from datasets import load_dataset      # HF datasets: удобная загрузка и разбиение датасета промоторов
from sklearn.metrics import (
    roc_auc_score, accuracy_score, confusion_matrix, roc_curve,
    f1_score, recall_score, precision_score
)
# метрики качества классификатора: ROC-AUC, accuracy, матрица ошибок, ROC-кривая

SEED = 42                              # фиксированный seed для воспроизводимости экспериментов
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)   # дополнительная инициализация для всех CUDA-девайсов

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # выбор устройства: GPU или CPU
print(f"Устройство: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Доступно памяти: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---

<a id="bio-questions"></a>

## 1. Биологические вопросы

В этом разделе важно не просто дать определение терминам, а показать, что вы понимаете биологический смысл задачи предсказания промоторов.

**Что ожидается в каждом вопросе:**
> 1. Сначала приведите ответ ИИ-ассистента.
> 2. Затем напишите свой ответ своими словами.
> 3. По возможности добавьте 1–2 уточнения: почему это важно именно для задачи предсказания промоторов или генерации последовательностей.

Оценивается прежде всего ваш собственный текст: он не должен быть дословной копией ответа ИИ, но должен быть корректным по существу.

#### Вопрос 1. Что такое промотор и где он находится в геноме? 

Опишите своими словами: что такое промотор, где он расположен относительно гена, и какую функцию выполняет.


**Ответ ИИ:**
*Напишите здесь ответ ИИ...*


**Ваш ответ:**
*Напишите здесь свой ответ...*


#### Вопрос 2. Что такое TSS (Transcription Start Site)? 

Что это за точка в геноме, что происходит именно в ней, и почему она важна для предсказания промоторов?


**Ответ ИИ:**
*Напишите здесь ответ ИИ...*


**Ваш ответ:**
*Напишите здесь свой ответ...*


#### Вопрос 3. Чем TATA-промоторы отличаются от non-TATA промоторов? 

Что такое TATA-бокс, у каких генов он встречается, и почему большинство промоторов человека не имеют TATA-бокса?


**Ответ ИИ:**
*Напишите здесь ответ ИИ...*


**Ваш ответ:**
*Напишите здесь свой ответ...*


#### Вопрос 4. Зачем нужно вычислительное предсказание промоторов? 

Какие практические задачи решает автоматическое определение промоторных областей в геноме?


**Ответ ИИ:**
*Напишите здесь ответ ИИ...*


**Ваш ответ:**
*Напишите здесь свой ответ...*


---

<a id="transformers-theory"></a>

## Раздел 2. Трансформеры в геномике (теория)

Если посмотреть на ДНК-последовательность, она выглядит как текст: `ATGCTAGCTAGCTA...`. Алфавит маленький (4 буквы вместо ~30 000 слов в NLP), но последовательности длинные и структурированные.

ДНК похожа на язык в нескольких смыслах:
- **Локальная структура:** рядом стоящие нуклеотиды образуют функциональные мотивы (например, TATAAA — сигнал промотора), как слова образуют фразы
- **Дальние зависимости:** регуляторные элементы могут быть за тысячи пар от гена, но влиять на его экспрессию — аналог дальних зависимостей в тексте
- **Контекст важен:** один и тот же мотив в разном контексте может иметь разные функции

#### DNABERT-2

Мы используем **DNABERT-2** (`zhihan1996/DNABERT-2-117M`) — модель на основе архитектуры MosaicBERT:
- **117M параметров**, 12 слоёв transformer
- **Посимвольная токенизация** (A, C, G, T → отдельные токены)
- **ALiBi** (Attention with Linear Biases) вместо стандартных positional embeddings
- **FlashAttention** для эффективных вычислений
- Предобучена на **135 геномах**

#### Что такое fine-tuning?

Fine-tuning — это процесс дообучения предобученной модели на конкретной задаче. Мы берём DNABERT-2 (которая "понимает" общую структуру ДНК) и обучаем её отличать промоторы от не-промоторов.

![Архитектура fine-tuning DNABERT-2](images/finetuning_arch.png)

Процесс: последовательность ДНК → токенизация → DNABERT-2 → CLS-токен (агрегированное представление) → Linear(hidden, 2) → Softmax → предсказание.

<a id="dnabert2-questions"></a>

### 2.1 Вопросы по DNABERT-2

#### Вопрос 1. Архитектура DNABERT-2

> **Задание:**
> - Опишите, какая базовая архитектура лежит в основе DNABERT-2.
> - Укажите порядок числа параметров модели.
> - Объясните, на каких последовательностях и в каком режиме модель предобучалась.
>
> **Ответ ИИ-ассистента:** [TODO: вставьте сюда сгенерированный ответ]
>
> **Ваш ответ своими словами:** [TODO: перефразируйте и дополните]


#### Вопрос 2. Токенизация в DNABERT-1 и DNABERT-2

> **Задание:**
> - Объясните, чем отличается `k-mer` токенизация в DNABERT-1 от токенизации в DNABERT-2.
> - Какие недостатки возникают у `k-mer` подхода?
> - Почему более гибкая токенизация полезна для длинных геномных последовательностей?
>
> **Ответ ИИ-ассистента:** [TODO]
>
> **Ваш ответ своими словами:** [TODO]


#### Вопрос 3. Что такое ALiBi?

> **Задание:**
> - Объясните идею `ALiBi (Attention with Linear Biases)`.
> - Сравните ALiBi с классическими positional encodings: sinusoidal и learned positional embeddings.
> - Почему ALiBi может быть удобен при работе с последовательностями переменной длины?
>
> **Ответ ИИ-ассистента:** [TODO]
>
> **Ваш ответ своими словами:** [TODO]


#### Вопрос 4. Что такое Flash Attention?

> **Задание:**
> - Опишите, какую проблему стандартного self-attention решает Flash Attention.
> - Как это влияет на использование памяти и скорость вычислений?
> - Почему это особенно важно для длинных последовательностей, в том числе геномных?
>
> **Ответ ИИ-ассистента:** [TODO]
>
> **Ваш ответ своими словами:** [TODO]


#### Вопрос 5. Что такое LoRA и зачем она нужна?

> **Задание:**
> - Объясните идею `LoRA (Low-Rank Adaptation)`.
> - Чем LoRA-fine-tuning отличается от полного fine-tuning?
> - Сравните подходы по памяти, числу обучаемых параметров, скорости и гибкости.
>
> **Ответ ИИ-ассистента:** [TODO]
>
> **Ваш ответ своими словами:** [TODO]


<a id="finetuning"></a>

## Задание 3: Fine-tuning DNABERT-2 (15 баллов)

В этом задании вы загрузите датасет промоторов, файнтюните DNABERT-2 как бинарный классификатор (промотор / не промотор) и оцените его качество.

<a id="data-loading"></a>

#### 3.1 Загрузка и исследование данных

Здесь вы подготовите датасет для задачи бинарной классификации: промотор / не-промотор.

**Что должно быть сделано в этом разделе (Definition of done):**

Минимально ожидается, что вы:
> 1. Загружаете датасет `InstaDeepAI/nucleotide_transformer_downstream_tasks` с конфигурацией `promoter_all`.
> 2. Получаете train/test split и выводите несколько примеров.
> 3. Проверяете названия полей, типы данных и распределение классов.
> 4. Проверяете длины последовательностей.
> 5. Отфильтровываете данные так, чтобы остались только последовательности длиной ровно `300`.
> 6. Коротко комментируете результат:
>    - насколько сбалансированы классы;
>    - есть ли последовательности «неправильной» длины;
>    - почему важно зафиксировать длину до токенизации и обучения.

**Подсказка:** перед обучением полезно явно проверить, что в последовательностях встречаются только допустимые символы ДНК (`A`, `C`, `G`, `T`), если в датасете нет дополнительных букв.

In [ ]:
# TODO (5 баллов)

TARGET_LEN = 300

train_dataset = None
test_dataset = None

# --- ваш код здесь ---

In [ ]:
from collections import Counter

assert train_dataset is not None, "train_dataset не загружен"
assert test_dataset is not None, "test_dataset не загружен"
assert "sequence" in train_dataset.features, "Нет поля 'sequence'"
assert "label" in train_dataset.features, "Нет поля 'label'"
assert set(train_dataset["label"]) == {0, 1}, "Метки должны быть 0 и 1"

# Проверка фильтрации
lengths = set(len(s) for s in train_dataset["sequence"])
assert lengths == {TARGET_LEN}, (
    f"Все последовательности должны быть длиной {TARGET_LEN}. "
    f"Найдены длины: {lengths}. Не забудьте отфильтровать датасет!"
)

print(f"Датасет загружен и отфильтрован!")
print(f"  Train: {len(train_dataset)} примеров")
print(f"  Test:  {len(test_dataset)} примеров")
print("Тесты пройдены!")

In [ ]:
# Фильтрация уже должна быть сделана выше.
# Далее будем работать с последовательностями длиной TARGET_LEN = 300.

<a id="tokenization"></a>

#### 3.2 Токенизация и DataLoader

В этом разделе вы подготовите данные в формате, пригодном для подачи в `DNABERT-2`.

**Что должно быть сделано в этом разделе (Definition of done):**

Минимально ожидается, что вы:
> 1. Загружаете токенизатор `zhihan1996/DNABERT-2-117M`.
> 2. Реализуете класс `PromoterDataset(Dataset)`, который:
>    - хранит HF-датасет и токенизатор;
>    - по индексу берёт строку последовательности и метку;
>    - токенизирует последовательность;
>    - возвращает словарь с ключами `input_ids`, `attention_mask`, `label`.
> 3. Создаёте `DataLoader` для train и test.
> 4. Проверяете shapes батча и типы тензоров.
>
> **Обратите внимание:**  
> - `label` должен быть целочисленным классом, пригодным для `CrossEntropyLoss`;  
> - батч должен иметь форму `(batch_size, seq_len_tokens)` для `input_ids` и `attention_mask`;  
> - разумно заранее зафиксировать `max_length` и поведение `padding` / `truncation`.

In [ ]:
# TODO

tokenizer = AutoTokenizer.from_pretrained(
    "zhihan1996/DNABERT-2-117M",
    trust_remote_code=True
)
print(f"Токенизатор загружен. Размер словаря: {tokenizer.vocab_size}")

BATCH_SIZE_FINETUNE = 32


class PromoterDataset(Dataset):
    """
    PyTorch Dataset для промоторных последовательностей.

    Должен:
    - Хранить HuggingFace-датасет и токенизатор
    - В __getitem__ токенизировать последовательность и вернуть словарь:
      {'input_ids': ..., 'attention_mask': ..., 'label': ...}
    """
    def __init__(self, hf_dataset, tokenizer, max_length=512):
        # TODO
        pass

    def __len__(self):
        # TODO
        pass

    def __getitem__(self, idx):
        # TODO
        pass


# TODO: создайте train_loader и test_loader

# --- ваш код здесь ---

In [ ]:
# Проверка Dataset и DataLoader
assert train_loader is not None, "train_loader не создан"
assert test_loader is not None, "test_loader не создан"

batch = next(iter(train_loader))

required_keys = {"input_ids", "attention_mask", "label"}
assert required_keys.issubset(batch.keys()), (
    f"Батч должен содержать ключи {required_keys}, получено: {set(batch.keys())}"
)

assert batch["input_ids"].ndim == 2, (
    f"input_ids должен иметь shape (batch_size, seq_len), получено {batch['input_ids'].shape}"
)
assert batch["attention_mask"].shape == batch["input_ids"].shape, (
    "attention_mask должен иметь тот же shape, что и input_ids"
)
assert batch["label"].ndim == 1, (
    f"label должен иметь shape (batch_size,), получено {batch['label'].shape}"
)

assert batch["input_ids"].shape[0] == BATCH_SIZE_FINETUNE, (
    f"Размер батча должен быть {BATCH_SIZE_FINETUNE}, получено {batch['input_ids'].shape[0]}"
)

print("Dataset и DataLoader работают корректно.")
print(f"input_ids shape: {batch['input_ids'].shape}")
print(f"attention_mask shape: {batch['attention_mask'].shape}")
print(f"label shape: {batch['label'].shape}")

<a id="model"></a>

#### 3.3 Модель с classification head (5 баллов)

Реализуйте класс `DNABERTClassifier`:
- Загружает DNABERT-2 как `self.bert`
- Добавляет `nn.Dropout` и `nn.Linear(self.bert.config.hidden_size, num_labels)` как classification head
- В `forward()` прогоняет последовательность через BERT, берёт представление первого токена последовательности (`last_hidden_state[:, 0, :]`), играющего роль sequence-level representation

In [ ]:
# TODO

class DNABERTClassifier(nn.Module):
    def __init__(self, num_labels=2, dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(
            "zhihan1996/DNABERT-2-117M",
            trust_remote_code=True
        )

        # TODO 
        
        pass  # удалите pass после реализации


    def forward(self, input_ids, attention_mask):
        # TODO: 
        
        pass  # удалите pass после реализации


print("Загружаем DNABERT-2 (117M параметров), первый раз скачивается ~470 MB...")
classifier = DNABERTClassifier(num_labels=2, dropout=0.1).to(device)
print(f"Модель создана. Параметров: {sum(p.numel() for p in classifier.parameters()):,}")


In [ ]:
classifier.eval()
with torch.no_grad():
    batch = next(iter(train_loader))
    out = classifier(
        batch["input_ids"].to(device),
        batch["attention_mask"].to(device)
    )
assert out is not None, "forward() вернул None"
assert out.shape == (BATCH_SIZE_FINETUNE, 2), (
    f"Ожидали shape ({BATCH_SIZE_FINETUNE}, 2), получили {out.shape}"
)
print(f" forward() работает корректно! Выход: {out.shape}")
print(f"  Пример logits: {out[0].cpu().numpy()}")
print(f"  Вероятности (softmax): {F.softmax(out[0], dim=-1).cpu().numpy()}")


<a id="training-functions"></a>

#### 3.4 Функции обучения

В этом разделе вы реализуете две базовые функции:

- `train_step` — один шаг обучения на батче;
- `eval_epoch` — полная оценка модели на валидационном / тестовом `DataLoader`.

**Что ожидается:**
> - в `train_step` корректно выполняются `forward`, вычисление `loss`, `backward`, `gradient clipping`, шаг оптимизатора и шаг scheduler;
> - в `eval_epoch` модель переводится в `eval()` режим;
> - по всей выборке собираются предсказания и считаются метрики хотя бы `loss`, `ROC-AUC`, `accuracy`.

**Подсказка:** для бинарной классификации с двумя логитами удобно брать вероятность положительного класса через `softmax(logits, dim=-1)[:, 1]`.

In [ ]:
def train_step(model, optimizer, scheduler, batch):
    """
    Один шаг обучения.

    Args:
        model: DNABERTClassifier
        optimizer: оптимизатор
        scheduler: lr scheduler (или None)
        batch: словарь с ключами 'input_ids', 'attention_mask', 'label'
    Returns:
        loss_value (float): значение loss на этом батче
    """
    # TODO:
    # 1. Переведите модель в режим обучения
    # 2. Перенесите данные на device
    # 3. Forward pass → вычислите cross-entropy loss
    # 4. Backward pass с gradient clipping (max_norm=1.0)
    # 5. Шаг оптимизатора и scheduler
    # --- ваш код здесь ---
    pass


@torch.no_grad()
def eval_epoch(model, loader):
    """
    Оценка модели на целом DataLoader.

    Returns:
        avg_loss (float), auc (float), accuracy (float)
    """
    # TODO:
    # 1. Переведите модель в eval mode
    # 2. Для каждого батча: forward pass, собирайте предсказания и метки
    # 3. Вычислите средний loss, ROC-AUC и accuracy
    # --- ваш код здесь ---
    pass

In [ ]:
# Smoke test для train_step / eval_epoch
optimizer_test = torch.optim.AdamW(classifier.parameters(), lr=1e-5)

batch = next(iter(train_loader))
loss_value = train_step(classifier, optimizer_test, scheduler=None, batch=batch)

assert isinstance(loss_value, float), f"train_step должен возвращать float, получено {type(loss_value)}"
assert np.isfinite(loss_value), f"loss должен быть конечным числом, получено {loss_value}"

val_loss, val_auc, val_acc = eval_epoch(classifier, test_loader)

assert isinstance(val_loss, float), "eval_epoch должен возвращать avg_loss типа float"
assert isinstance(val_auc, float), "eval_epoch должен возвращать auc типа float"
assert isinstance(val_acc, float), "eval_epoch должен возвращать accuracy типа float"

assert np.isfinite(val_loss), f"val_loss некорректен: {val_loss}"
assert 0.0 <= val_auc <= 1.0, f"AUC должен быть в [0, 1], получено {val_auc}"
assert 0.0 <= val_acc <= 1.0, f"Accuracy должен быть в [0, 1], получено {val_acc}"

print("train_step и eval_epoch работают корректно.")
print(f"train loss: {loss_value:.4f}")
print(f"val loss:   {val_loss:.4f}")
print(f"val auc:    {val_auc:.4f}")
print(f"val acc:    {val_acc:.4f}")

<a id="training-loop"></a>

#### 3.5 Обучающий цикл (5 баллов)

Реализуйте обучающий цикл:
- Эпохи: 3 (этого достаточно для файнтюна)
- В конце каждой эпохи выводите: эпоха, train_loss, val_loss, val_auc, val_acc

In [ ]:
# TODO 
N_EPOCHS = 3
LR_FINETUNE = None

# TODO: создать optimizer = torch.optim.AdamW(...)

# TODO: создать scheduler 

finetune_history = {"train_loss": [], "val_loss": [], "val_auc": [], "val_acc": []}

# TODO: Цикл по эпохам
# for epoch in range(N_EPOCHS):
#     Цикл по батчам -> train_step(...)
#     eval_epoch(classifier, test_loader)
#     Логирование и print

# --- ваш код здесь ---


<a id="evaluation"></a>

#### 3.6 Оценка модели (5 баллов)

Посчитайте метрики на тестовой выборке и визуализируйте результаты.

In [ ]:
# TODO (5 баллов)
# 1. Получите предсказания модели на test_loader
# 2. Вычислите метрики: ROC-AUC, Accuracy, F1-score, Recall, Precision
# 3. Постройте визуализации:
#    - ROC-кривую
#    - Confusion matrix (heatmap)
#    - Кривые обучения (train_loss, val_auc по эпохам из finetune_history)

# --- ваш код здесь ---

In [ ]:
# Проверка: убедитесь, что модель достигает приемлемого качества
# Запустите после реализации оценки выше

assert final_auc > 0.75, f"AUC={final_auc:.3f} слишком низкий. Проверьте обучение."
print("Модель обучена с достаточным качеством (AUC > 0.75)!")

In [ ]:
os.makedirs("./dnabert_promoter", exist_ok=True)
torch.save(classifier.state_dict(), "./dnabert_promoter/classifier.pt")
print("Модель сохранена в ./dnabert_promoter/classifier.pt")

# Освобождаем память GPU перед RL
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"Память GPU очищена. Зарезервировано: {torch.cuda.memory_reserved(0)/1e9:.2f} GB")
else:
    print("CUDA недоступна, пропускаем очистку GPU-памяти.")

<a id="lora"></a>

#### 3.7 LoRA Fine-tuning

Теперь сравним два подхода:

1. **Полный fine-tuning** — обучаются все параметры модели.
2. **LoRA fine-tuning** — основные веса заморожены, а обучаются только небольшие low-rank адаптеры.

**Что должно быть сделано в этом разделе (Definition of done):**

Минимально ожидается, что вы:
> 1. Устанавливаете и импортируете `peft`.
> 2. Создаёте новую базовую модель для эксперимента с LoRA.
> 3. Настраиваете `LoraConfig`:
>    - выбираете `r`, `lora_alpha`, `lora_dropout`;
>    - указываете `target_modules`.
> 4. Обучаете LoRA-версию модели.
> 5. Сравниваете полный fine-tuning и LoRA по:
>    - числу обучаемых параметров;
>    - времени обучения;
>    - качеству на тесте (`AUC`, `Accuracy`, желательно `F1`).
> 6. Делаете короткий вывод: когда LoRA особенно выгодна, а когда полный fine-tuning может быть предпочтительнее.


- В базовом варианте применяйте LoRA к линейным проекциям в attention-блоках (например, query/value или q_proj/v_proj — в зависимости от того, как названы модули в конкретной реализации модели).

- Не ограничивайтесь только classification head: это слишком слабый вариант PEFT и
не даёт содержательного сравнения с полным fine-tuning.

In [ ]:
!pip install -q peft

from peft import get_peft_model, LoraConfig, TaskType
import time

# TODO: Создайте базовую модель (новый экземпляр DNABERTClassifier)
# lora_base_model = DNABERTClassifier(num_labels=2).to(device)

# TODO: Создайте LoRA-конфигурацию
# Параметры для экспериментов: r (rank), lora_alpha, target_modules
# lora_config = LoraConfig(
#     r=...,           # ранг (попробуйте 4, 8, 16, 32)
#     lora_alpha=...,  # масштабирующий коэффициент
#     target_modules=[...],  # какие слои адаптировать
#     lora_dropout=0.1,
#     bias="none",
#     task_type=TaskType.SEQ_CLS
# )

# TODO: Оберните модель в LoRA и выведите число обучаемых параметров
# lora_model = get_peft_model(lora_base_model, lora_config)
# lora_model.print_trainable_parameters()

# --- ваш код здесь ---

In [ ]:
# TODO: Обучите LoRA-модель
# Используйте те же train_step / eval_epoch что и для полного файнтюна
# Замерьте время обучения с помощью time.time()

# start_time = time.time()
# ... обучение ...
# lora_train_time = time.time() - start_time

# --- ваш код здесь ---

In [ ]:
# TODO: Сравнительный анализ
# 1. Таблица: Full fine-tune vs LoRA
#    - Число обучаемых параметров
#    - Время обучения (секунды)
#    - ROC-AUC, Accuracy, F1 на тесте
# 2. Попробуйте разные значения rank (4, 8, 16, 32) — как меняется качество?
# 3. Визуализируйте: bar-chart сравнения метрик для full vs LoRA (разные rank)

# --- ваш код здесь ---

In [ ]:
# Проверка LoRA-конфигурации
assert lora_model is not None, "LoRA-модель не создана"

trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in lora_model.parameters())

assert trainable_params > 0, "В LoRA-модели нет обучаемых параметров"
assert trainable_params < all_params, (
    "LoRA должна обучать только часть параметров, а не все параметры модели"
)

print("LoRA-модель настроена корректно.")
print(f"Trainable params: {trainable_params:,}")
print(f"All params:       {all_params:,}")
print(f"Trainable ratio:  {100 * trainable_params / all_params:.4f}%")

<a id="rl-theory"></a>

## Раздел 4. Введение в Reinforcement Learning

Перед тем как перейти к реализации, разберём ключевые концепции RL, которые вам понадобятся.


#### 4.1 Что такое RL?

Reinforcement Learning — это обучение через взаимодействие. Агент не получает готовых ответов (как в supervised learning). Вместо этого он:
1. Наблюдает **состояние** среды
2. Выбирает **действие**
3. Получает **награду** (feedback от среды)
4. Обновляет свою **политику**, чтобы в будущем получать больше наград

Формально это описывается как **Markov Decision Process (MDP)**:
- $\mathcal{S}$ — множество состояний
- $\mathcal{A}$ — множество действий
- $P(s'|s, a)$ — вероятности переходов
- $R(s, a)$ — функция награды
- $\gamma$ — фактор дисконтирования (в нашем случае $\gamma = 1$)

#### 4.2 Политика (Policy)

Политика $\pi_\theta(a|s)$ — это стратегия агента: вероятностное распределение над действиями в данном состоянии.

- **Стохастическая:** $a \sim \pi_\theta(\cdot|s)$ — сэмплируем действие
- **Детерминистическая:** $a = \mu_\theta(s)$ — фиксированный выбор

В нашей задаче политика **стохастическая**: LSTM выдаёт вероятности 4 нуклеотидов.

#### 4.3 Return (доход)

Полный доход эпизода: $G_t = \sum_{k=t}^{T} \gamma^{k-t} r_k$

В нашей задаче: $\gamma = 1$, а $r_k = 0$ для всех $k < T$, и $r_T = \text{DNABERT\_score}(\text{sequence})$.

Значит $G_t = r_T$ для всех $t$ — все шаги получают одинаковый return (sparse reward).

#### 4.4 Policy Gradient Theorem

Цель: максимизировать $J(\theta) = \mathbb{E}_{\pi_\theta}[G_0]$.

Теорема о градиенте политики:
$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\left[ G_t \cdot \nabla_\theta \log \pi_\theta(a_t|s_t) \right]$$

#### 4.5 REINFORCE

Простейший алгоритм policy gradient:
1. Собрать эпизод $\tau = (s_0, a_0, r_0, ..., s_T, a_T, r_T)$
2. Для каждого шага $t$: вычислить $G_t$
3. Обновить: $\theta \leftarrow \theta + \alpha \cdot G_t \cdot \nabla_\theta \log \pi_\theta(a_t|s_t)$

#### 4.6 Проблема: высокая дисперсия

REINFORCE работает, но градиенты очень шумные. Решения:

1. **Baseline:** вычитаем среднее: $\nabla J \approx (G_t - b) \cdot \nabla \log \pi$
   - $b$ — baseline, обычно скользящее среднее наград
   - Не вносит bias, но снижает variance

2. **Entropy bonus:** добавляем $H(\pi) = -\sum_a \pi(a) \log \pi(a)$ к целевой функции
   - Поощряет исследование
   - Предотвращает слишком раннюю сходимость к одному действию

#### 4.7 Итоговая формула loss

$$\mathcal{L} = -\frac{1}{B} \sum_{i=1}^{B} \left[ A_i \cdot \sum_{t=0}^{T-1} \log \pi_\theta(a_t^i|s_t^i) \right] - \beta \cdot \bar{H}$$

где:
- $A_i = r_i - \text{baseline}$ (advantage)
- $\bar{H}$ — средняя энтропия по всем шагам и эпизодам
- $\beta = 0.05$ — коэффициент entropy bonus

#### 4.8 Схема эпизода

Каждый эпизод — генерация последовательности из 300 нуклеотидов:

![Схема эпизода RL](images/rl_episode.png)

Награда выдаётся **только в конце** — это sparse reward. На всех промежуточных шагах $r_t = 0$.

<a id="rl-env"></a>

## 5. RL-среда

<a id="dna-env"></a>

#### 5.1 Класс DNAEnvironment

В этом разделе вы реализуете простую среду, в которой агент по одному символу строит ДНК-последовательность фиксированной длины.



**Интерфейс среды:**
- `reset()` — сбрасывает эпизод и возвращает начальное состояние;
- `step(action: int)` — выполняет действие и возвращает `(next_state, reward, done, info)`;
- `get_sequence()` — возвращает текущую последовательность как строку.

**Соглашения:**
- пространство действий: `0=A`, `1=C`, `2=G`, `3=T`;
- до завершения эпизода награда равна `0.0`;
- финальная награда выдаётся только в конце эпизода и равна оценке reward-модели для сгенерированной последовательности;
- эпизод заканчивается, когда длина последовательности достигает `seq_length`.

**Что должно быть сделано в этом разделе (Definition of done):**
> 1. Среда корректно инициализируется.
> 2. После `reset()` длина последовательности равна нулю.
> 3. Каждый вызов `step()` добавляет ровно один нуклеотид.
> 4. Награда до конца эпизода равна `0.0`.
> 5. На последнем шаге среда вызывает `compute_rewards([sequence])[0]`.
> 6. Реализованы базовые проверки корректности через `assert`.

**Подсказка:** это пример среды со `sparse reward`, поэтому обучение дальше будет существенно шумнее, чем в supervised learning.

In [ ]:
SEQ_LEN = 300
NUCLEOTIDES = ['A', 'C', 'G', 'T']
NUC_TO_IDX = {n: i for i, n in enumerate(NUCLEOTIDES)}
print(f"Длина эпизода: {SEQ_LEN} шагов")
print(f"Нуклеотиды: {NUCLEOTIDES}")


In [ ]:
# TODO 
class DNAEnvironment:
    """
    Среда для пошаговой генерации ДНК-последовательности.

    Атрибуты:
        seq_length (int): целевая длина последовательности
        sequence (list): список выбранных нуклеотидов (строки 'A','C','G','T')
        step_count (int): текущий шаг эпизода
    """

    NUCLEOTIDES = NUCLEOTIDES

    def __init__(self, seq_length=SEQ_LEN):
        # TODO:
        pass

    def reset(self):
        """
        Сбросить среду для нового эпизода.
        Returns:
            None (нет предыдущего нуклеотида)
        """
        # TODO:
        pass

    def step(self, action: int):
        """
        Сделать один шаг: добавить нуклеотид с индексом action.
        Args:
            action (int): индекс нуклеотида (0=A, 1=C, 2=G, 3=T)
        Returns:
            next_state: action (int) — последний выбранный нуклеотид
            reward (float): 0.0 пока не конец, иначе score модели
            done (bool): True если последовательность заполнена
            info (dict): {'sequence': строка, 'step': номер шага}
        """
        # TODO: 
        pass

    def get_sequence(self) -> str:
        """Вернуть текущую последовательность как строку"""
        # TODO
        pass


#### Вопросы для самопроверки: RL-среда

1. Почему награда выдаётся только в конце эпизода (sparse reward), а не на каждом шаге? Какие альтернативные схемы награды можно было бы использовать и в чём их проблемы?
2. Какова размерность пространства действий и пространства состояний в нашей среде? Как это соотносится с обычными задачами RL (например, Atari-игры)?
3. Почему начальное состояние — `None`, а не какой-то конкретный нуклеотид? Что это означает для первого шага агента?

**Ответ ИИ:**
*Напишите здесь ответ ИИ...*

**Ваш ответ:**
*Напишите здесь свой ответ...*


<a id="reward-fn"></a>

#### 5.2 Функция награды (5 баллов)

Реализуйте загрузку reward-модели и функцию `compute_rewards` для батчевого вычисления наград.

In [ ]:
# Загружаем файнтюненую модель для использования как reward function
print("Загружаем файнтюненую модель как reward function...")

reward_tokenizer = AutoTokenizer.from_pretrained(
    "zhihan1996/DNABERT-2-117M",
    trust_remote_code=True
)

# TODO: Создайте экземпляр DNABERTClassifier и загрузите сохранённые веса
# Не забудьте перевести модель в режим оценки
reward_model = None  # замените


@torch.no_grad()
def compute_rewards(sequences: list, batch_size: int = 16) -> list:
    """
    Вычисляет вероятность промотора для списка ДНК-последовательностей.

    Args:
        sequences: список строк ДНК (длина 300)
        batch_size: размер батча для инференса
    Returns:
        list of float: вероятности [0, 1] для каждой последовательности
    """
    # TODO:
    # 1. Обрабатывайте последовательности батчами для эффективности
    # 2. Токенизируйте батч, пропустите через reward_model
    # 3. Извлеките вероятность класса 1 (промотор) через softmax
    # --- ваш код здесь ---
    pass

In [ ]:
# Тест среды
env = DNAEnvironment(seq_length=SEQ_LEN)
state = env.reset()
assert state is None, "reset() должен возвращать None"
assert env.step_count == 0, "step_count должен быть 0 после reset"

done = False
for t in range(SEQ_LEN):
    action = np.random.randint(4)
    next_state, reward, done, info = env.step(action)
    if t < SEQ_LEN - 1:
        assert reward == 0.0, f"На шаге {t} reward должен быть 0, получили {reward}"
        assert done == False, f"done должен быть False на шаге {t}"
    else:
        assert done == True, "done должен быть True на последнем шаге"
        assert 0.0 <= reward <= 1.0, f"reward должен быть в [0,1], получили {reward}"

assert len(env.get_sequence()) == SEQ_LEN, "Длина последовательности неверная"
print(f" DNAEnvironment работает корректно!")
print(f"  Финальная награда (случайная последовательность): {reward:.4f}")
print(f"  Последовательность: {env.get_sequence()[:30]}...")


In [ ]:
# Проверка reward_model и compute_rewards
assert reward_model is not None, "reward_model не инициализирован"

test_sequences = [
    "A" * SEQ_LEN,
    "C" * SEQ_LEN,
    "ACGT" * (SEQ_LEN // 4)
]

rewards = compute_rewards(test_sequences, batch_size=2)

assert isinstance(rewards, list), f"compute_rewards должен возвращать list, получено {type(rewards)}"
assert len(rewards) == len(test_sequences), (
    f"Длина rewards должна быть {len(test_sequences)}, получено {len(rewards)}"
)
assert all(isinstance(r, float) for r in rewards), "Все элементы rewards должны быть float"
assert all(0.0 <= r <= 1.0 for r in rewards), "Все rewards должны быть в диапазоне [0, 1]"

print("reward_model и compute_rewards работают корректно.")
print("Пример наград:", rewards)

---

<a id="lstm-policy"></a>

## Задание 6: Policy Network — LSTMPolicy (10 баллов)

Реализуйте LSTM-политику. На каждом шаге $t$ она получает **one-hot вектор** предыдущего нуклеотида (или нулевой вектор на первом шаге), обновляет скрытое состояние LSTM и выдаёт распределение вероятностей над 4 нуклеотидами.

![Архитектура LSTMPolicy](images/lstm_policy.png)

**Что нужно реализовать:**
- `__init__()`: рекуррентный и выходной слои
- `forward(x, hidden)`: один шаг LSTM → логиты
- `init_hidden(device)`: нулевое начальное состояние
- `get_action(prev_action, hidden, device)`: one-hot → forward → Categorical → сэмпл

In [ ]:
# TODO (10 баллов)

class LSTMPolicy(nn.Module):
    """
    Стохастическая политика на основе LSTM для генерации ДНК-последовательностей.

    На каждом шаге принимает один нуклеотид (one-hot) и выдаёт
    распределение над следующим нуклеотидом.
    """
    N_NUCLEOTIDES = 4  # A, C, G, T

    def __init__(self, hidden_size: int = 128, num_layers: int = 2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # TODO: Определите рекуррентный слой и выходной слой
        # Вход рекуррентного слоя: размер 4 (one-hot нуклеотид)
        # Выход: распределение над 4 нуклеотидами
        # --- ваш код здесь ---
        pass

    def forward(self, x, hidden):
        """
        Один шаг LSTM.

        Args:
            x: тензор shape (1, 1, 4) — one-hot вектор нуклеотида
            hidden: tuple (h_n, c_n) — скрытое состояние
        Returns:
            logits: тензор shape (1, 4) — логиты для каждого нуклеотида
            hidden: обновлённое скрытое состояние
        """
        # TODO
        # --- ваш код здесь ---
        pass

    def init_hidden(self, device) -> tuple:
        """
        Инициализирует скрытое состояние нулями.

        Returns:
            (h_0, c_0): кортеж из двух тензоров shape (num_layers, 1, hidden_size)
        """
        # TODO
        # --- ваш код здесь ---
        pass

    def get_action(self, prev_action, hidden, device):
        """
        Выбирает следующий нуклеотид из стохастической политики.

        Args:
            prev_action: int или None (если первый шаг)
            hidden: текущее скрытое состояние
            device: torch device
        Returns:
            action (int): индекс нуклеотида 0-3
            log_prob (Tensor): log-вероятность выбранного действия
            entropy (Tensor): энтропия распределения
            hidden: обновлённое скрытое состояние
        """
        # TODO:
        # 1. Создайте one-hot вектор из prev_action (или нулевой, если None)
        # 2. Forward pass через LSTM
        # 3. Softmax → Categorical distribution → сэмпл
        # --- ваш код здесь ---
        pass


print("Создаём LSTMPolicy...")
policy = LSTMPolicy(hidden_size=128, num_layers=2).to(device)
n_params = sum(p.numel() for p in policy.parameters())
print(f"  Параметров: {n_params:,}")

In [ ]:
policy.eval()
with torch.no_grad():
    hidden = policy.init_hidden(device)
    assert hidden is not None, "init_hidden вернул None"
    assert len(hidden) == 2, "hidden должен быть кортежем (h_n, c_n)"
    assert hidden[0].shape == (2, 1, 128), f"h_n shape неверный: {hidden[0].shape}"

    action, log_prob, entropy, new_hidden = policy.get_action(None, hidden, device)
    assert isinstance(action, int) and 0 <= action <= 3, f"action должен быть 0-3, получили {action}"
    assert log_prob.shape == (), f" log_prob должен быть скаляром"
    assert entropy.shape == (), f" entropy должен быть скаляром"
    assert entropy.item() > 0, " Энтропия должна быть > 0"

    # Проверка что вероятности суммируются в 1
    x = torch.zeros(1, 1, 4, device=device)
    logits, _ = policy(x, hidden)
    probs = F.softmax(logits[0], dim=-1)
    assert abs(probs.sum().item() - 1.0) < 1e-5, f"Вероятности не суммируются в 1: {probs.sum()}"

print(" LSTMPolicy работает корректно!")
print(f"  Пример действия: {NUCLEOTIDES[action]}")
print(f"  log_prob: {log_prob.item():.4f}")
print(f"  entropy: {entropy.item():.4f} (max possible: {math.log(4):.4f})")


#### Вопросы для самопроверки: LSTM-политика

1. Почему для генерации последовательности используется LSTM, а не обычная feedforward-сеть? Какое свойство задачи делает рекуррентную архитектуру подходящей?
2. Что хранится в скрытом состоянии LSTM (h и c)? Какую информацию о ранее сгенерированных нуклеотидах может "запоминать" агент?
3. Почему на вход подаётся one-hot вектор предыдущего нуклеотида, а не, например, embedding? В каких случаях embedding был бы полезнее?
4. Как связан выход Linear-слоя с вероятностями нуклеотидов? Зачем нужен Categorical distribution?

**Ответ ИИ:**
*Напишите здесь ответ ИИ...*

**Ваш ответ:**
*Напишите здесь свой ответ...*


<a id="reinforce"></a>

## Задание 7: Алгоритм REINFORCE (20 баллов)

<a id="trajectory"></a>

#### 7.1 Сбор траектории (5 баллов)

Реализуйте функцию `collect_episode`, которая разыгрывает один полный эпизод.

In [ ]:
# TODO

def collect_episode(env: DNAEnvironment, policy: LSTMPolicy, device) -> tuple:
    """
    Разыгрывает один эпизод генерации последовательности.

    На каждом шаге:
    1. policy.get_action(prev_action, hidden, device) -> action, log_prob, entropy, hidden
    2. env.step(action) -> _, reward, done, _

    Args:
        env: DNAEnvironment
        policy: LSTMPolicy
        device: torch.device
    Returns:
        log_probs (list[Tensor]): log-вероятности выбранных действий (длина SEQ_LEN)
        reward (float): финальная награда
        sequence (str): сгенерированная последовательность
        entropies (list[Tensor]): энтропии на каждом шаге
    """
    policy.train()
    state = env.reset()
    hidden = policy.init_hidden(device)

    log_probs = []
    entropies = []
    prev_action = None

    # TODO: 
    # --- ваш код здесь ---

    return log_probs, reward, env.get_sequence(), entropies


<a id="baseline"></a>

#### 7.2 Baseline и Advantage

В классическом `REINFORCE` градиенты часто имеют большую дисперсию. Чтобы сделать обучение стабильнее, введём `baseline` — экспоненциальное скользящее среднее наград.

Будем использовать:
- `baseline` как оценку «типичной» награды;
- `advantage = reward - baseline`;
- нормализацию `advantage` внутри батча.

**Что ожидается:**
> - реализован класс `ExponentialMovingAverage`;
> - baseline корректно обновляется по батчу наград;
> - advantages считаются относительно baseline;
> - advantages дополнительно нормализуются, чтобы уменьшить шум в обучении.

In [ ]:
class ExponentialMovingAverage:
    """
    Baseline для REINFORCE: экспоненциальное скользящее среднее наград.

    Формула обновления: value = (1 - alpha) * value + alpha * mean(rewards)

    Методы:
        update(rewards) -> обновляет среднее, возвращает новое значение
        get() -> возвращает текущее значение (0.0 если ещё не было обновлений)
    """
    def __init__(self, alpha=0.1):
        self.alpha = alpha
        self.value = None

    def update(self, rewards: list) -> float:
        # TODO: вычислите среднее наград и обновите self.value по формуле EMA
        # --- ваш код здесь ---
        pass

    def get(self) -> float:
        # TODO: верните текущее значение или 0.0 если value is None
        # --- ваш код здесь ---
        pass

#### Вопросы для самопроверки: Награды и Baseline

1. Зачем нужен baseline в REINFORCE? Что происходит с дисперсией градиентов без него?
2. Почему мы используем экспоненциальное скользящее среднее, а не просто среднее по всем прошлым наградам? В чём преимущество EMA?
3. Что такое advantage и почему нормализация advantages помогает стабильности обучения?
4. Как связаны entropy bonus и exploration? Что произойдёт, если убрать entropy из loss?

**Ответ ИИ:**
*Напишите здесь ответ ИИ...*

**Ваш ответ:**
*Напишите здесь свой ответ...*


In [ ]:
# TODO 

def compute_advantages(rewards: list, baseline: ExponentialMovingAverage) -> list:
    """
    Обновляет baseline и вычисляет advantage для батча эпизодов.

    Advantage_i = reward_i - baseline_value

    Дополнительно нормализуйте advantages (вычтите среднее, разделите на std + 1e-8).
    Нормализация помогает стабильности обучения.

    Args:
        rewards: список наград за эпизоды [r_1, r_2, ..., r_B]
        baseline: объект ExponentialMovingAverage
    Returns:
        advantages: список float
    """
    # TODO: 
    # --- ваш код здесь ---
    pass


<a id="policy-update"></a>

#### 7.3 Обновление политики (10 баллов)

Реализуйте функцию `update_policy`, которая считает loss и делает шаг оптимизатора.

In [ ]:
# TODO 

def update_policy(
    optimizer,
    log_probs_batch: list,
    advantages: list,
    entropies_batch: list,
    entropy_coef: float = 0.05
) -> float:
    """
    Один шаг обновления политики по REINFORCE с baseline и entropy bonus.

    Loss = -1/B * Σᵢ [advantage_i * Σₜ log π(aₜ|sₜ)] - entropy_coef * mean_entropy

    Знак минус потому что PyTorch минимизирует, а мы максимизируем доход.

    Args:
        optimizer: torch оптимизатор
        log_probs_batch: list of lists — log-вероятности для каждого эпизода
        advantages: list of float — advantages для каждого эпизода
        entropies_batch: list of lists — энтропии для каждого эпизода
        entropy_coef: вес entropy bonus
    Returns:
        loss_value (float): значение loss
    """
    B = len(log_probs_batch)
    total_loss = torch.tensor(0.0, device=device)

    # TODO: 

    # --- ваш код здесь ---

    return total_loss.item()


In [ ]:
policy.train()
env_test = DNAEnvironment(seq_length=10)  # короткий эпизод для теста

log_probs, reward, seq, entropies = collect_episode(env_test, policy, device)
assert len(log_probs) == 10, f"Ожидали 10 log_probs, получили {len(log_probs)}"
assert len(entropies) == 10, f"Ожидали 10 энтропий, получили {len(entropies)}"
assert len(seq) == 10, f"Ожидали 10 символов, получили {len(seq)}"
assert all(c in NUCLEOTIDES for c in seq), f"Последовательность содержит недопустимые символы: {seq}"
print("collect_episode работает!")

# Тест update_policy — loss должен меняться после шага
optimizer_test = torch.optim.Adam(policy.parameters(), lr=1e-3)
baseline_test = ExponentialMovingAverage()

params_before = [p.clone() for p in policy.parameters()]

# Собираем несколько эпизодов
all_lp, all_rew, all_ent = [], [], []
for _ in range(4):
    lp, r, _, ent = collect_episode(env_test, policy, device)
    all_lp.append(lp); all_rew.append(r); all_ent.append(ent)

advs = compute_advantages(all_rew, baseline_test)
loss_val = update_policy(optimizer_test, all_lp, advs, all_ent)

assert loss_val is not None, "update_policy вернул None"
params_after = [p.clone() for p in policy.parameters()]
params_changed = any(not torch.equal(b, a) for b, a in zip(params_before, params_after))
assert params_changed, "Параметры политики не изменились после обновления!"
print(f"✓ update_policy работает! Loss: {loss_val:.4f}")
print(f"✓ Параметры обновились")


<a id="baselines"></a>

## Задание 8: Базовые стратегии (10 баллов)

Для осмысленного сравнения нужны базовые стратегии, с которыми мы будем сравнивать RL-агента.

<a id="random-baseline"></a>

#### 8.1 Случайная генерация (3 балла)

Самый простой baseline: генерировать последовательности случайным образом.

In [ ]:
# TODO 
def random_baseline(n_sequences: int = 200, seq_length: int = SEQ_LEN) -> dict:
    """
    Генерирует случайные ДНК-последовательности и оценивает их.

    Args:
        n_sequences: количество последовательностей
        seq_length: длина каждой последовательности
    Returns:
        dict с ключами:
            'sequences': list of str
            'rewards': list of float (из compute_rewards)
            'mean_reward': float
            'max_reward': float
    """
    # TODO:
    pass

print("Оцениваем случайную стратегию...")
random_results = random_baseline(n_sequences=200)
print(f"Случайная генерация:")
print(f"  Средняя награда: {random_results['mean_reward']:.4f}")
print(f"  Максимальная награда: {random_results['max_reward']:.4f}")


<a id="freq-baseline"></a>

#### 8.2 Частотный baseline (greedy, 7 баллов)

Более умный baseline: строим матрицу частот нуклеотидов по позициям из **реальных промоторов** тренировочного датасета, затем генерируем последовательности, сэмплируя из этого распределения.

Это даёт "знающий" baseline — он использует информацию о реальных промоторах, которой у RL-агента нет. Если RL превзойдёт этот baseline — это хороший результат.

In [ ]:
# TODO [8.2]: Частотный baseline (7 баллов)

def build_frequency_matrix(sequences: list, seq_length: int = SEQ_LEN) -> np.ndarray:
    """
    Строит матрицу частот нуклеотидов по позициям.

    Args:
        sequences: список промоторных последовательностей из train_dataset
        seq_length: длина последовательности
    Returns:
        freq_matrix: np.ndarray shape (seq_length, 4)
            freq_matrix[i, j] = вероятность нуклеотида j на позиции i
            Сумма по строке должна равняться 1
    """
    # TODO: подсчитайте частоту каждого нуклеотида на каждой позиции
    # и нормализуйте так, чтобы сумма по строке равнялась 1
    # --- ваш код здесь ---
    pass


def frequency_baseline(freq_matrix: np.ndarray, n_sequences: int = 200,
                       seq_length: int = SEQ_LEN) -> dict:
    """
    Генерирует последовательности, сэмплируя нуклеотиды согласно частотной матрице.

    Args:
        freq_matrix: матрица частот (seq_length, 4)
        n_sequences: количество последовательностей
    Returns:
        dict с ключами: 'sequences', 'rewards', 'mean_reward', 'max_reward'
    """
    # TODO: для каждой последовательности сэмплируйте нуклеотид на каждой позиции
    # согласно распределению из freq_matrix
    # --- ваш код здесь ---
    pass


# Извлекаем промоторные последовательности из train_dataset
promoter_seqs = [ex["sequence"] for ex in train_dataset if ex["label"] == 1]
print(f"Промоторных последовательностей: {len(promoter_seqs)}")

freq_matrix = build_frequency_matrix(promoter_seqs)
assert freq_matrix is not None, "build_frequency_matrix вернула None"
assert freq_matrix.shape == (SEQ_LEN, 4), f"Ожидали shape ({SEQ_LEN}, 4), получили {freq_matrix.shape}"
assert np.allclose(freq_matrix.sum(axis=1), 1.0, atol=1e-5), "Строки не нормализованы!"
print(f"Матрица частот построена: shape {freq_matrix.shape}")

freq_results = frequency_baseline(freq_matrix)
assert freq_results is not None, "frequency_baseline вернула None"
print(f"Частотный baseline: mean={freq_results['mean_reward']:.4f}, max={freq_results['max_reward']:.4f}")

#### Вопросы для самопроверки: Базовые стратегии

Прежде чем двигаться дальше, ответьте на вопросы:

1. Почему важно сравнивать RL-агента с базовыми стратегиями? Что мы хотим показать?
2. В чём "нечестность" частотного baseline по сравнению с RL-агентом? Какой информацией он пользуется, которой нет у агента на старте обучения?
3. Если RL-агент не превосходит частотный baseline, значит ли это, что RL не работает? Почему?

**Ответ ИИ:**
*Напишите здесь ответ ИИ...*

**Ваш ответ:**
*Напишите здесь свой ответ...*


<a id="training-analysis"></a>

## 9. Обучение и анализ

<a id="rl-training"></a>

### 9.1 Обучающий цикл REINFORCE

Ниже расположен основной цикл обучения policy network.

**Что должно быть сделано в этом разделе (Definition of done):**

Минимально ожидается, что вы:
> 1. Инициализируете политику, оптимизатор, baseline и среду.
> 2. На каждом обновлении собираете батч эпизодов.
> 3. Считаете `advantages`.
> 4. Обновляете политику через `update_policy`.
> 5. Сохраняете историю обучения:
>    - `mean_reward`,
>    - `max_reward`,
>    - `min_reward`,
>    - `loss`,
>    - `mean_entropy`,
>    - `baseline`.
> 6. Периодически логируете прогресс.
>
> **Коротко прокомментируйте после запуска:**  
> растёт ли средняя награда, снижается ли энтропия, насколько стабильно обучение.

In [ ]:
N_UPDATES   = 300    # обновлений политики
BATCH_SIZE  = 16     # эпизодов на обновление
LR_POLICY   = 3e-4   # learning rate
ENTROPY_COEF = 0.05  # вес entropy bonus
EVAL_EVERY  = 25     # логировать каждые N обновлений

# Переинициализируем политику и оптимизатор
policy = LSTMPolicy(hidden_size=128, num_layers=2).to(device)
optimizer = torch.optim.Adam(policy.parameters(), lr=LR_POLICY)
baseline = ExponentialMovingAverage(alpha=0.05)
env = DNAEnvironment(seq_length=SEQ_LEN)

history = {
    "mean_reward": [], "max_reward": [], "min_reward": [],
    "loss": [], "mean_entropy": [], "baseline": []
}

print(f"Начинаем обучение: {N_UPDATES} обновлений × {BATCH_SIZE} эпизодов")
print(f"Всего эпизодов: {N_UPDATES * BATCH_SIZE}")
print("=" * 60)

for update in tqdm(range(1, N_UPDATES + 1), desc="Training"):
    all_log_probs, all_rewards, all_sequences, all_entropies = [], [], [], []

    # Собираем батч эпизодов
    for _ in range(BATCH_SIZE):
        log_probs, reward, sequence, entropies = collect_episode(env, policy, device)
        all_log_probs.append(log_probs)
        all_rewards.append(reward)
        all_sequences.append(sequence)
        all_entropies.append(entropies)

    # Advantage = reward - baseline (нормализованное)
    advantages = compute_advantages(all_rewards, baseline)

    # Обновление политики
    loss_val = update_policy(optimizer, all_log_probs, advantages, all_entropies, ENTROPY_COEF)

    # Логирование
    history["mean_reward"].append(sum(all_rewards) / len(all_rewards))
    history["max_reward"].append(max(all_rewards))
    history["min_reward"].append(min(all_rewards))
    history["loss"].append(loss_val)
    history["baseline"].append(baseline.get())
    mean_ent = sum(e.item() for ep in all_entropies for e in ep) / (BATCH_SIZE * SEQ_LEN)
    history["mean_entropy"].append(mean_ent)

    if update % EVAL_EVERY == 0:
        print(f"  Update {update:3d}/{N_UPDATES} | "
              f"Reward: {history['mean_reward'][-1]:.4f} "
              f"(max={history['max_reward'][-1]:.4f}) | "
              f"Baseline: {baseline.get():.4f} | "
              f"Entropy: {mean_ent:.3f}")

print("=" * 60)
print(f"  Обучение завершено!")
print(f"  Финальная средняя награда: {history['mean_reward'][-1]:.4f}")
print(f"  Финальная максимальная награда: {history['max_reward'][-1]:.4f}")


<a id="visualization"></a>

### 9.2 Визуализация обучения (5 баллов)

Постройте кривые обучения и сравните все методы.

In [ ]:
# TODO 
# Постройте графики:
# 1. Кривые обучения RL-агента: mean_reward и max_reward vs update
#    Добавьте горизонтальные линии для random_results['mean_reward']
#    и freq_results['mean_reward']
# 2. Энтропия политики vs update (должна снижаться по мере обучения)
# 3. Baseline vs update (должен расти)
# 4. Box-plot или violin-plot сравнения: random | freq_baseline | rl_best_200


# --- ваш код здесь ---


<a id="seq-analysis"></a>

### 9.3 Анализ последовательностей (10 баллов)

Изучите, что именно RL-агент научился генерировать.

In [ ]:
# TODO [9.3]: Анализ лучших последовательностей (10 баллов)

# Шаг 1: Сгенерируйте 500 последовательностей обученной политикой
# Шаг 2: Отберите топ-50 по reward

# Шаг 3: Посчитайте для топ-50 и 50 случайных:
#   a) GC-состав: (G + C) / длина
#   b) Содержание TATA-бокса: найдите TATAAA с помощью str.find()
#   c) Частоты нуклеотидов (A, C, G, T в %)
# Выведите сравнительную таблицу

# Шаг 4: Постройте position frequency matrix для топ-50 RL-последовательностей
# (аналогично build_frequency_matrix)
# --- ваш код здесь ---


<a id="discussion"></a>

### 9.4 Дискуссионные вопросы

Ответьте на вопросы ниже. Здесь важно не только описать наблюдения, но и связать их с ограничениями reward-модели, baseline-стратегий и самого RL-подхода.

#### Вопрос А. Почему RL-агент может быть лучше случайной генерации, но хуже частотного baseline?

> **Задание:**
> - Объясните, почему даже простое RL-обучение может улучшать reward по сравнению со случайной генерацией.
> - Почему частотный baseline может оказаться очень сильным?
> - Можно ли по этому сравнению сделать вывод, что RL «не работает»?
>
> **Ответ ИИ-ассистента:** [TODO]
>
> **Ваш ответ своими словами:** [TODO]

#### Вопрос Б. Ограничения reward-модели

> **Задание:**
> - Какие ограничения есть у подхода, где награда определяется предсказанием классификатора?
> - Может ли агент «взломать» reward-модель и начать генерировать последовательности, которые выглядят хорошими только для классификатора?
> - Можно ли считать такие последовательности биологически осмысленными без дополнительной проверки?
>
> **Ответ ИИ-ассистента:** [TODO]
>
> **Ваш ответ своими словами:** [TODO]

#### Вопрос В. Как можно улучшить пайплайн?

> **Задание:**
> - Предложите хотя бы 2 конкретных улучшения текущего пайплайна.
> - Для каждого улучшения объясните, какую проблему оно решает.
> - Это могут быть улучшения в reward-функции, policy network, RL-алгоритме, baseline, анализе результатов или биологической валидации.
>
> **Ответ ИИ-ассистента:** [TODO]
>
> **Ваш ответ своими словами:** [TODO]

<a id="ppo"></a>

## Бонус: Proximal Policy Optimization (PPO) (+10 баллов)



#### Что такое PPO и зачем он нужен?

Проблема REINFORCE: если advantage большой, мы делаем большой шаг градиента, что может "сломать" политику (переобучиться на один удачный батч). PPO решает это через **ограничение размера обновления**.

PPO использует **clipped surrogate objective**:

$$\mathcal{L}^{\text{CLIP}}(\theta) = \mathbb{E}_t \left[ \min\left( r_t(\theta) \cdot A_t,\; \text{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon) \cdot A_t \right) \right]$$

где:
- $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)} = \exp(\log\pi_{\text{new}} - \log\pi_{\text{old}})$ — отношение вероятностей
- $\varepsilon = 0.2$ — типичное значение клипирования
- $A_t$ — advantage

**Идея:** если advantage положительный (действие было хорошим), мы хотим увеличить вероятность этого действия, но не больше чем в $(1+\varepsilon)$ раз. Аналогично для отрицательного advantage.

In [ ]:
# TODO: 

def ppo_update(
    policy,
    optimizer,
    log_probs_batch_old: list,   # старые log-prob (detached)
    log_probs_batch_new_fn,      # функция, пересчитывающая log-prob через текущую политику
    advantages: list,
    entropies_batch: list,
    entropy_coef: float = 0.05,
    clip_eps: float = 0.2,
    n_epochs: int = 4
) -> float:
    """
    PPO обновление политики с clipped surrogate objective.

    Ключевое отличие от REINFORCE:
    - Вычисляем ratio = exp(log_pi_new - log_pi_old)
    - Клипируем ratio в [1-clip_eps, 1+clip_eps]
    - Берём min(ratio * adv, clipped_ratio * adv)
    - Повторяем n_epochs раз на одном батче

    Args:
        log_probs_batch_old: старые log-вероятности (из сбора данных, detach)
        advantages: advantages (нормализованные)
        clip_eps: параметр клиппинга ε
        n_epochs: количество эпох обновления на одном батче
    """
    # TODO: 
    pass

# Подсказка: для пересчёта log_probs нужно проиграть эпизод заново
# или хранить все (state, action) пары и пересчитать по ним
# Проще: хранить actions и пересчитать log_pi(a|s) через forward

print("Шаблон PPO готов — реализуйте ppo_update()")


In [ ]:
# TODO: Обучение PPO и сравнение с REINFORCE
# 1. Обучить PPO-агента с теми же гиперпараметрами
# 2. Построить сравнительный график mean_reward vs update для обоих методов
# 3. Прокомментировать разницу в скорости сходимости и стабильности

# --- ваш код здесь ---
